 # **1. Import Dependencies and Page Setup**

 Modified to remove 'ee' and add 'geopy' for offline city geocoding.

In [1]:
%%writefile app.py

# ==========================================
# IMPORT DEPENDENCIES
# ==========================================
import os
import pandas as pd
import numpy as np
import folium
from streamlit_folium import folium_static
import joblib
import streamlit as st
from geopy.geocoders import Nominatim



Writing app.py


 # **2. Page Configuration**

 Cleaned block. Google Earth Engine authentication and initialization removed.

In [2]:
%%writefile -a app.py

# ==========================================
# PAGE CONFIGURATION
# ==========================================
st.set_page_config(page_title="Carbon Prediction Dashboard", layout="wide")



Appending to app.py


 # **3. Spatial Customization Parameters**

 Preserved for layout configuration.

In [3]:
%%writefile -a app.py

# ==========================================
# COLOR PALETTES & GRAPHICS CONFIGURATION
# ==========================================
dicionario_cores = {
    1: "#32a65e", 3: "#1f8d49", 4: "#7dc975", 5: "#04381d", 6: "#026975",
    49: "#02d659", 10: "#ad975a", 11: "#519799", 12: "#d6bc74", 32: "#fc8114",
    29: "#ffaa5f", 50: "#ad5100", 13: "#d89f5c", 14: "#FFFFB2", 15: "#edde8e",
    18: "#E974ED", 19: "#C27BA0", 39: "#f5b3c8", 20: "#db7093", 40: "#c71585",
    62: "#ff69b4", 41: "#f54ca9", 36: "#d082de", 46: "#d68fe2", 47: "#9932cc",
    48: "#e6ccff", 9: "#7a5900", 21: "#ffefc3", 22: "#d4271e", 23: "#ffa07a",
    24: "#d4271e", 30: "#9c0027", 25: "#db4d4f", 26: "#0000FF", 33: "#2532e4",
    31: "#091077", 27: "#ffffff"
}

dicionario_classes = {
    1: "Floresta", 3: "Formação Florestal", 4: "Formação Savânica", 5: "Mangue",
    6: "Floresta Alagável (beta)", 49: "Restinga Arborizada", 10: "Formação Natural não Florestal",
    11: "Campo Alagado e Área Pantanosa", 12: "Formação Campestre", 32: "Apicum",
    29: "Afloramento Rochoso", 50: "Restinga Herbácea", 13: "Outras Formações não Florestais",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura", 19: "Lavoura Temporária",
    39: "Soja", 20: "Cana", 40: "Arroz", 62: "Algodão (beta)", 41: "Outras Lavouras Temporárias",
    36: "Lavoura Perene", 46: "Café", 47: "Citrus", 48: "Outras Lavouras Perenes",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Área não Vegetada", 23: "Praia, Duna e Areal",
    24: "Área Urbanizada", 30: "Mineração", 25: "Outras Áreas não Vegetadas", 26: "Corpo D'água",
    33: "Rio, Lago e Oceano", 31: "Aquicultura", 27: "Não observado"
}



Appending to app.py


 # **4. Load Data & ML Model (Cached)**

In [4]:
%%writefile -a app.py

# ==========================================
# LOAD DATA & ML MODEL (CACHED)
# ==========================================
@st.cache_data
def load_csv():
    csv_path = "/content/drive/MyDrive/Colab Notebooks/files/output/final/data_integration_all_areas_final.csv"
    return pd.read_csv(csv_path)

@st.cache_resource
def load_ml_model():
    model_path = '/content/drive/MyDrive/Colab Notebooks/files/output/final/modelo_arvore_carbono.pkl'
    try:
        return joblib.load(model_path)
    except Exception as e:
        st.sidebar.error(f"⚠️ Model Load Error: {e}")
        return None

all_farms_df = load_csv()
ml_model = load_ml_model()



Appending to app.py


 # **5. Dashboard Interface & Spatial Processing (Approximation Mode)**

 Replaced Earth Engine imagery processing with live city geocoding and calculated area circles.

In [5]:
%%writefile -a app.py

# ==========================================
# SIDEBAR / USER INTERFACE
# ==========================================
st.sidebar.header("📍 Location Filters")

list_state = sorted(list(all_farms_df['state_name'].unique()))
state_name = st.sidebar.selectbox('State:', list_state)

list_cities = sorted(list(all_farms_df[all_farms_df['state_name'] == state_name]['city_name'].unique()))
city_name = st.sidebar.selectbox('City:', list_cities)

list_farms = sorted(list(all_farms_df[all_farms_df['city_name'] == city_name]['farm'].unique()))
farm_id = st.sidebar.selectbox('Farm (ID):', list_farms)

# ==========================================
# METRICS & AI INFERENCE EXECUTION
# ==========================================
if farm_id:
    data = all_farms_df[all_farms_df['farm'] == farm_id].iloc[0]
    
    st.title(f"🌱 Predicted Carbon Dashboard - Farm {farm_id}")
    st.markdown("---")
    
    ai_prediction_str = "Unavailable"
    if ml_model is not None:
        try:
            features = [[data['area_size']]] 
            prediction_value = ml_model.predict(features)[0]
            ai_prediction_str = f"{prediction_value:.4f}"
        except Exception as ml_err:
            ai_prediction_str = "Inference Error"

    col1, col2, col3, col4, col5 = st.columns(5)
    col1.metric("Area (ha)", f"{data['area_size']:.2f}")
    col2.metric("Biome", data['biome_name'])
    col3.metric("tCO2/ha (Historical Reality)", f"{data['balance_CO2_ha']:.4f}")
    col4.metric("tCO2/ha (Predicted AI)", ai_prediction_str)
    col5.metric("Total CO2", f"{data['balance_CO2_area']:.2f}")
    
    st.markdown("---")
    
    # ==========================================
    # LOCAL GEOPROCESSING & FOLIUM MAP (NO GEE)
    # ==========================================
    with st.spinner('Calculating approximate farm location...'):
        try:
            # 1. Initialize local geolocator to find the city center coordinates
            geolocator = Nominatim(user_agent="carbon_dashboard_ufjf")
            location_query = f"{city_name}, {state_name}, Brazil"
            location = geolocator.geocode(location_query, timeout=10)
            
            if location:
                center_lat = location.latitude
                center_lon = location.longitude
            else:
                # Default coordinates (Center of Brazil) if geocoding fails
                center_lat = -14.2350
                center_lon = -51.9253

            # 2. Create standard Folium base map centered on the selected city
            M = folium.Map(location=[center_lat, center_lon], zoom_start=11, control_scale=True)

            # 3. Calculate circle radius in meters based on the 'area_size' column (Hectares)
            # Formula: 1 ha = 10,000 m². Radius = sqrt(Area / pi)
            area_ha = float(data['area_size'])
            radius_meters = np.sqrt((area_ha * 10000) / np.pi)

            # 4. Draw the approximation circle onto the map layout
            folium.Circle(
                location=[center_lat, center_lon],
                radius=radius_meters,
                color="#32a65e",
                fill=True,
                fill_color="#7dc975",
                fill_opacity=0.5,
                popup=f"Farm {farm_id} - Estimated limits based on {area_ha:.2f} hectares."
            ).add_to(M)

            # 5. Add a marker pointing to the reference city center
            folium.Marker(
                location=[center_lat, center_lon],
                popup=f"Reference City Center: {city_name}",
                icon=folium.Icon(color="green", icon="info-sign")
            ).add_to(M)

            # 6. Render the map using the static transmission block
            folium_static(M, width=700, height=500)

            # 7. Spatial information alert
            st.info(f"ℹ️ Map Display: Showing a calculated proportional boundary circle of {radius_meters:.1f} meters around the center of {city_name} to represent the farm size.")

        except Exception as e:
            st.error(f"⚠️ Error rendering the approximation map: {e}")



Appending to app.py


 # **6. Cloud Deployment Execution**

In [6]:
# 1. Install local dependencies adding 'geopy'
!pip install -q streamlit==1.29.0 folium streamlit-folium geopy

# 2. Install web tunnel tool
!npm install -g localtunnel

# 3. Fetch the Password/IP for authentication
import urllib
print("==========================================================")
print("🔑 SENHA/IP PARA ACESSAR O SITE:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("==========================================================")

# 4. Run the dashboard
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇==========================================================
🔑 SENHA/IP PARA ACESSAR O SITE: 34.19.73.122
your url is: https://public-donkeys-film.loca.lt



  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8502
  External URL: http://34.19.73.122:8502

  Stopping...
^C
